# SpaceBros quick-start: NISAR over Delhi
Runs locally or in Google Colab. Steps 1–2 need no login; step 3 needs a free NASA Earthdata account.

In [ ]:
# Colab only: get the repo + libraries (replace with our repo URL)
# !git clone https://github.com/<owner>/spacebros-nisar.git
# %cd spacebros-nisar
# !pip install -q -r requirements.txt

In [ ]:
import sys, json, pandas as pd
sys.path.insert(0, 'src')
import config
print('Study area:', config.AOI_NAME, config.AOI_BBOX)

## 1. What NISAR data exists over the study area?

In [ ]:
!python src/nisar_search.py
cat = pd.read_csv(f'data/catalogue/{config.AOI_NAME}_scenes.csv')
cat.groupby(['level','direction','track']).size()

## 2. Quick-look time-lapse (public images, no login)

In [ ]:
!python src/fetch_browse.py --track 106
from IPython.display import Image
Image(filename=f'data/browse/{config.AOI_NAME}_T106/timelapse.gif')

## 3. Real backscatter (needs Earthdata login)
earthaccess will ask for your username/password, or read them from `~/.netrc`. Don't put passwords in this notebook.

In [ ]:
!python src/gcov_subset.py --track 106 --start 2026-06-01 --end 2026-10-31 --pols HHHH HVHV

In [ ]:
import glob, rasterio, matplotlib.pyplot as plt, numpy as np
files = sorted(glob.glob(f'data/subsets/{config.AOI_NAME}/*_HVHV.tif'))
fig, axes = plt.subplots(1, min(4, len(files)), figsize=(16, 4))
for ax, f in zip(np.atleast_1d(axes), files[:4]):
    with rasterio.open(f) as s: ax.imshow(s.read(1), cmap='gray', vmin=-25, vmax=-5)
    ax.set_title(f.split('/')[-1][:10]); ax.axis('off')

## 4. Time series: mean backscatter per date

In [ ]:
rows = []
for f in sorted(glob.glob(f'data/subsets/{config.AOI_NAME}/*.tif')):
    if 'change_' in f: continue
    date, track, pol = f.split('/')[-1][:-4].split('_')
    with rasterio.open(f) as s: rows.append(dict(date=date, pol=pol, mean_db=float(np.nanmean(s.read(1)))))
ts = pd.DataFrame(rows).pivot(index='date', columns='pol', values='mean_db')
ts.plot(marker='o', figsize=(10, 4), ylabel='mean backscatter (dB)'); ts

## 5. Change map between two dates

In [ ]:
# pick two dates from the same track
!python src/change_map.py data/subsets/delhi/2026-06-09_T106_HHHH.tif data/subsets/delhi/2026-09-13_T106_HHHH.tif